<a href="https://colab.research.google.com/github/langelh/dm_2016325_2026_2/blob/main/tallerSQL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import urllib.request
import os
import sqlite3
import pandas as pd
import math

In [5]:
url_chinook = "https://raw.githubusercontent.com/lerocha/chinook-database/master/ChinookDatabase/DataSources/Chinook_Sqlite.sqlite"

ruta_db = "chinook.db"

if not os.path.exists(ruta_db):
    urllib.request.urlretrieve(url_chinook, ruta_db)
    print(f"Base de datos descargada en: {ruta_db}")
else:
    print(f"La base de datos ya existe: {ruta_db}")

La base de datos ya existe: chinook.db


In [6]:
con = sqlite3.connect("chinook.db")

cur = con.cursor()

print("Conexión realizada correctamente.")

Conexión realizada correctamente.


In [7]:
cur.execute("""
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name;
""")

tablas = [fila[0] for fila in cur.fetchall()]

print("Tablas disponibles:")
print(tablas)

Tablas disponibles:
['Album', 'Artist', 'Customer', 'Employee', 'Genre', 'Invoice', 'InvoiceLine', 'MediaType', 'Playlist', 'PlaylistTrack', 'Track']


In [8]:
cur.execute("PRAGMA table_info(Track);")

estructura_track = pd.DataFrame(
    cur.fetchall(),
    columns=["cid", "name", "type", "notnull", "dflt_value", "pk"]
)

estructura_track

,cid,name,type,notnull,dflt_value,pk
0,0,TrackId,INTEGER,1,None,1
1,1,Name,NVARCHAR(200),1,None,0
2,2,AlbumId,INTEGER,0,None,0
3,3,MediaTypeId,INTEGER,1,None,0
4,4,GenreId,INTEGER,0,None,0
5,5,Composer,NVARCHAR(220),0,None,0
6,6,Milliseconds,INTEGER,1,None,0
7,7,Bytes,INTEGER,0,None,0
8,8,UnitPrice,"NUMERIC(10,2)",1,None,0


ejercicio 1

In [9]:
pregunta_1 = pd.read_sql_query("""
    SELECT COUNT(*) AS total_pistas
    FROM Track;
""", con)

pregunta_1

,total_pistas
0,3503


In [10]:
pregunta_2 = pd.read_sql_query("""
    SELECT COUNT(*) AS total_generos
    FROM Genre;
""", con)

pregunta_2

,total_generos
0,25


In [11]:
pregunta_3 = pd.read_sql_query("""
    SELECT
        Genre.Name AS genero,
        COUNT(Track.TrackId) AS numero_pistas,
        ROUND(AVG(Track.Milliseconds) / 60000.0, 2)
            AS duracion_promedio_minutos
    FROM Genre
    INNER JOIN Track
        ON Genre.GenreId = Track.GenreId
    GROUP BY Genre.GenreId, Genre.Name
    ORDER BY numero_pistas DESC;
""", con)

pregunta_3

,genero,numero_pistas,duracion_promedio_minutos
0,Rock,1297,4.73
1,Latin,579,3.88
2,Metal,374,5.16
3,Alternative & Punk,332,3.91
4,Jazz,130,4.86
5,TV Shows,93,35.75
6,Blues,81,4.51
7,Classical,74,4.90
8,Drama,64,42.92
9,R&B/Soul,61,3.67


In [12]:
pregunta_4 = pd.read_sql_query("""
    SELECT
        Track.Name AS pista,
        Album.Title AS album,
        ROUND(Track.Milliseconds / 60000.0, 2)
            AS duracion_minutos
    FROM Track
    INNER JOIN Album
        ON Track.AlbumId = Album.AlbumId
    ORDER BY Track.Milliseconds DESC
    LIMIT 5;
""", con)

pregunta_4

,pista,album,duracion_minutos
0,Occupation / Precipice,"Battlestar Galactica, Season 3",88.12
1,Through a Looking Glass,"Lost, Season 3",84.81
2,"Greetings from Earth, Pt. 1","Battlestar Galactica (Classic), Season 1",49.34
3,The Man With Nine Lives,"Battlestar Galactica (Classic), Season 1",49.28
4,"Battlestar Galactica, Pt. 2","Battlestar Galactica (Classic), Season 1",49.27


In [13]:
pregunta_5 = pd.read_sql_query("""
    SELECT
        COUNT(*) AS total_pistas,

        SUM(
            CASE
                WHEN UnitPrice > 0.99 THEN 1
                ELSE 0
            END
        ) AS pistas_mayor_099,

        ROUND(
            100.0 *
            SUM(
                CASE
                    WHEN UnitPrice > 0.99 THEN 1
                    ELSE 0
                END
            ) / COUNT(*),
            2
        ) AS porcentaje
    FROM Track;
""", con)

pregunta_5

,total_pistas,pistas_mayor_099,porcentaje
0,3503,213,6.08


In [29]:
con = sqlite3.connect("chinook.db")
cur = con.cursor()

pregunta_6 = pd.read_sql_query("""
    SELECT
        ROUND(AVG(Milliseconds), 2) AS media,
        MIN(Milliseconds) AS minimo,
        MAX(Milliseconds) AS maximo,
        ROUND(
            AVG(Milliseconds * Milliseconds)
            - AVG(Milliseconds) * AVG(Milliseconds),
            2
        ) AS varianza
    FROM Track;
""", con)

pregunta_6

,media,minimo,maximo,varianza
0,393599.21,1071,5286953,2.861491e+11


ejercicio 2

In [30]:
pregunta_7 = pd.read_sql_query("""
    SELECT
        ROUND(
            SUM(
                InvoiceLine.UnitPrice *
                InvoiceLine.Quantity
            ),
            2
        ) AS ingresos_totales,

        ROUND(
            SUM(
                InvoiceLine.UnitPrice *
                InvoiceLine.Quantity
            )
            / COUNT(DISTINCT Invoice.InvoiceId),
            2
        ) AS promedio_por_factura
    FROM Invoice
    INNER JOIN InvoiceLine
        ON Invoice.InvoiceId = InvoiceLine.InvoiceId;
""", con)

pregunta_7

,ingresos_totales,promedio_por_factura
0,2328.6,5.65


In [31]:
pregunta_8 = pd.read_sql_query("""
    SELECT
        strftime('%Y', InvoiceDate) AS año,
        COUNT(*) AS numero_facturas
    FROM Invoice
    GROUP BY año
    ORDER BY año ASC;
""", con)

pregunta_8

,año,numero_facturas
0,2021,83
1,2022,83
2,2023,83
3,2024,83
4,2025,80


In [17]:
pregunta_9 = pd.read_sql_query("""
    SELECT
        Invoice.BillingCountry AS pais,

        COUNT(DISTINCT Invoice.InvoiceId)
            AS numero_facturas,

        ROUND(
            SUM(
                InvoiceLine.UnitPrice *
                InvoiceLine.Quantity
            ),
            2
        ) AS ingreso_total,

        ROUND(
            SUM(
                InvoiceLine.UnitPrice *
                InvoiceLine.Quantity
            )
            / COUNT(DISTINCT Invoice.InvoiceId),
            2
        ) AS promedio_por_factura

    FROM Invoice

    INNER JOIN InvoiceLine
        ON Invoice.InvoiceId = InvoiceLine.InvoiceId

    GROUP BY Invoice.BillingCountry

    ORDER BY ingreso_total DESC

    LIMIT 5;
""", con)

pregunta_9

,pais,numero_facturas,ingreso_total,promedio_por_factura
0,USA,91,523.06,5.75
1,Canada,56,303.96,5.43
2,France,35,195.10,5.57
3,Brazil,35,190.10,5.43
4,Germany,28,156.48,5.59


In [18]:
totales_facturas = pd.read_sql_query("""
    SELECT
        Invoice.InvoiceId AS factura,

        SUM(
            InvoiceLine.UnitPrice *
            InvoiceLine.Quantity
        ) AS total_factura

    FROM Invoice

    INNER JOIN InvoiceLine
        ON Invoice.InvoiceId = InvoiceLine.InvoiceId

    GROUP BY Invoice.InvoiceId;
""", con)

totales_facturas

,factura,total_factura
0,1,1.98
1,2,3.96
2,3,5.94
3,4,8.91
4,5,13.86
...,...,...
407,408,3.96
408,409,5.94
409,410,8.91
410,411,13.86


In [19]:
resultado_varianza = pd.read_sql_query("""
    SELECT
        AVG(total_factura * total_factura)
        - AVG(total_factura) * AVG(total_factura)
        AS varianza

    FROM (
        SELECT
            Invoice.InvoiceId,

            SUM(
                InvoiceLine.UnitPrice *
                InvoiceLine.Quantity
            ) AS total_factura

        FROM Invoice

        INNER JOIN InvoiceLine
            ON Invoice.InvoiceId = InvoiceLine.InvoiceId

        GROUP BY Invoice.InvoiceId
    );
""", con)

resultado_varianza

,varianza
0,22.463404


In [20]:
varianza = resultado_varianza["varianza"].iloc[0]

print("Varianza:", varianza)

Varianza: 22.46340351116976


In [21]:
desviacion_estandar = math.sqrt(varianza)

print("Desviación estándar:", desviacion_estandar)

Desviación estándar: 4.739557311729626


In [22]:
ingresos_mes = pd.read_sql_query("""
    SELECT
        strftime('%m', Invoice.InvoiceDate) AS mes,

        ROUND(
            AVG(totales.total_factura),
            2
        ) AS ingreso_promedio

    FROM Invoice

    INNER JOIN (
        SELECT
            InvoiceId,

            SUM(
                UnitPrice * Quantity
            ) AS total_factura

        FROM InvoiceLine

        GROUP BY InvoiceId
    ) AS totales

    ON Invoice.InvoiceId = totales.InvoiceId

    GROUP BY mes

    ORDER BY ingreso_promedio DESC;
""", con)

ingresos_mes

,mes,ingreso_promedio
0,04,6.00
1,09,5.95
2,01,5.92
3,06,5.75
4,02,5.67
5,08,5.66
6,03,5.57
7,10,5.52
8,05,5.52
9,11,5.48


In [23]:
mayor_ingreso = ingresos_mes.iloc[0]

print("Mes con mayor ingreso promedio:")
print(mayor_ingreso)

Mes con mayor ingreso promedio:
mes                  04
ingreso_promedio    6.0
Name: 0, dtype: object


In [24]:
menor_ingreso = ingresos_mes.iloc[-1]

print("Mes con menor ingreso promedio:")
print(menor_ingreso)

Mes con menor ingreso promedio:
mes                  12
ingreso_promedio    5.4
Name: 11, dtype: object
